Packages:

In [1]:
import pandas as pd
import numpy as np
import wrds

Data sourcing:

In [2]:
db = wrds.Connection(wrsd_username="lucadesj")

Enter your WRDS username [lucad]: lucadesj
Enter your password: ········


WRDS recommends setting up a .pgpass file.


Create .pgpass file now [y/n]?:  y


Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [3]:
# Extract fundamentals annual
# Table : comp.funda

compustat_query = """
    SELECT
        gvkey, datadate, fyear, indfmt,
        conm,
        -- Compte de résultat
        sale, ebit, oibdp, dp, ni, txt, xint, oancf, capx, cogs, revt,
        -- Bilan
        at, ceq, dltt, dlc, che, lct,
        -- Actions & prix
        csho, prcc_f,
        -- Intangible
        xrd, xsga
    FROM comp.funda
    WHERE fyear BETWEEN 2000 AND 2025
      AND indfmt  = 'INDL'
      AND datafmt = 'STD'
      AND popsrc  = 'D'
      AND consol  = 'C'
      AND sale > 0
      AND at   > 0
      AND ceq  > 0
"""


db_comp = db.raw_sql(compustat_query, date_cols=['datadate'])
db_comp = db_comp.drop_duplicates(["gvkey", "fyear"])
db_comp.head()

,gvkey,datadate,fyear,indfmt,conm,sale,ebit,oibdp,dp,ni,...,at,ceq,dltt,dlc,che,lct,csho,prcc_f,xrd,xsga
0,001004,2001-05-31,2000,INDL,AAR CORP,874.255,45.79,64.367,18.577,18.531,...,701.854,340.212,179.987,13.652,13.809,125.392,26.937,14.0,<NA>,96.077
1,001010,2000-12-31,2000,INDL,ACF INDUSTRIES INC,443.8,117.0,178.7,61.7,79.9,...,3794.5,985.2,1866.8,176.1,113.0,<NA>,0.015,<NA>,<NA>,32.4
2,001013,2000-10-31,2000,INDL,ADC TELECOMMUNICATIONS INC,3287.9,523.9,670.1,146.2,868.1,...,3970.5,2912.7,16.5,28.5,1354.2,1041.3,770.3,21.375,338.0,1050.7
3,001019,2000-12-31,2000,INDL,AFA PROTECTIVE SYSTEMS INC,42.504,3.415,5.443,2.028,1.762,...,28.638,13.184,1.12,0.666,1.818,5.39,0.166,183.0,<NA>,12.554
4,001021,2000-06-30,2000,INDL,AFP IMAGING CORP,25.367,-0.028,0.672,0.7,-0.808,...,11.608,4.455,4.875,0.195,0.434,2.278,9.271,0.49,0.477,8.189


In [4]:
# Extract Monthly Price
# Table : crsp.msf 

crsp_query = """
   SELECT
    m.permno,
    m.date,
    m.prc,
    m.ret,
    m.retx,   
    m.shrout,
    n.shrcd,
    n.exchcd,
    d.dlret,
    d.dlstcd
FROM crsp.msf AS m
JOIN crsp.msenames AS n
    ON m.permno = n.permno
    AND m.date BETWEEN n.namedt AND n.nameendt
LEFT JOIN crsp.msedelist AS d
    ON m.permno = d.permno
    AND date_trunc('month', m.date) = date_trunc('month', d.dlstdt)
WHERE n.shrcd IN (10, 11)        -- Actions ordinaires uniquement
    AND n.exchcd IN (1, 2, 3)    -- NYSE, AMEX, et NASDAQ
    AND m.date >= '2000-01-01'
"""
db_crsp = db.raw_sql(crsp_query, date_cols=['date'])
db_crsp['date'] = db_crsp['date'] + pd.offsets.MonthEnd(0) #Ramène chaque date à la fin du mois
db_crsp.head()

,permno,date,prc,ret,retx,shrout,shrcd,exchcd,dlret,dlstcd
0,10001,2000-01-31,8.125,-0.044118,-0.044118,2450.0,11,3,<NA>,<NA>
1,10001,2000-02-29,8.25,0.015385,0.015385,2450.0,11,3,<NA>,<NA>
2,10001,2000-03-31,-8.0,-0.015758,-0.030303,2464.0,11,3,<NA>,<NA>
3,10001,2000-04-30,-8.09375,0.011719,0.011719,2464.0,11,3,<NA>,<NA>
4,10001,2000-05-31,-7.90625,-0.023166,-0.023166,2464.0,11,3,<NA>,<NA>


In [5]:
print(db_comp.shape)
print(db_crsp.shape)

(178243, 26)
(1294973, 10)


In [6]:
#Link table :
link_query = """
    SELECT gvkey, lpermno AS permno, linkdt, linkenddt
    FROM crsp.ccmxpf_lnkhist
    WHERE linktype IN ('LC', 'LU')
      AND linkprim IN ('P', 'C')
"""

db_link = db.raw_sql(link_query, date_cols=['linkdt', 'linkenddt'])
db_link['linkenddt'] = db_link['linkenddt'].fillna(pd.Timestamp('2099-12-31')) #Remplace les NaN par une date fictive future (lien encore actif)

db_link.head()

,gvkey,permno,linkdt,linkenddt
0,001000,25881.0,1970-11-13,1978-06-30
1,001001,10015.0,1983-09-20,1986-07-31
2,001002,10023.0,1972-12-14,1973-06-05
3,001003,10031.0,1983-12-07,1989-08-16
4,001004,54594.0,1972-04-24,2099-12-31


In [7]:
#Panel final

#Prise en compte du décalage entre les données et la date de clôture de l'année fiscale 
db_comp['avail_date'] = (db_comp['datadate'] + pd.DateOffset(months=6)) + pd.offsets.MonthEnd(0)

#Conservation des dates valides uniquement
crsp_linked = db_crsp.merge(db_link, on='permno', how='inner')

crsp_linked = crsp_linked[
    (crsp_linked['date'] >= crsp_linked['linkdt']) &
    (crsp_linked['date'] <= crsp_linked['linkenddt'])].drop(columns=['linkdt', 'linkenddt']) #Ne garde uniquement les valeurs comprises dans l'interval de validité du lien des "keys"


#Liaison entre chaque mois du CRSP et le dernier exercice Compustat dispo 

crsp_linked['date'] = pd.to_datetime(crsp_linked['date'])
db_comp['avail_date'] = pd.to_datetime(db_comp['avail_date'])

crsp_linked = crsp_linked.sort_values(['gvkey', 'date'])
db_comp = db_comp.sort_values(['gvkey', 'avail_date'])

In [8]:
import psutil
print(f"RAM utilisée : {psutil.virtual_memory().percent}%")
print(f"Swap utilisé : {psutil.swap_memory().percent}%")

RAM utilisée : 27.2%
Swap utilisé : 0.0%


In [9]:
# 1. Merge exact sur gvkey d'abord
temp = crsp_linked.merge(
    db_comp[['gvkey', 'avail_date', 'fyear', 'conm', 'indfmt', 'ni', 'ceq', 'oancf', 'sale', 
             'ebit', 'oibdp', 'dp', 'txt', 'xint', 'capx', 'revt','cogs', 'at', 'dltt', 
             'dlc', 'che', 'lct', 'csho', 'prcc_f', 'xrd', 'xsga']],
    on='gvkey', 
    how='left', 
    suffixes=('', '_comp')
)

# 2. Garder SEULEMENT les lignes où date CRSP <= date dispo comptes
temp = temp[temp['date'] <= temp['avail_date']]

# 3. Pour chaque (gvkey, date CRSP), prendre le compte LE PLUS RECENT disponible
panel = (temp
    .sort_values(['gvkey', 'date', 'avail_date'])
    .groupby(['gvkey', 'date'])
    .first()  # Premier = le plus récent avail_date
    .reset_index()
)

# Nettoyer les doublons colonnes
panel = panel.drop(columns=['avail_date_comp'], errors='ignore')

print(f"Panel : {panel.shape[0]:,} obs — {panel['permno'].nunique():,} titres")


Panel : 1,189,590 obs — 10,823 titres


In [10]:
panel.head()

,gvkey,date,permno,prc,ret,retx,shrout,shrcd,exchcd,dlret,...,cogs,at,dltt,dlc,che,lct,csho,prcc_f,xrd,xsga
0,001004,2000-01-31,54594,17.6875,-0.009199,-0.013937,27181.0,11,1,<NA>,...,713.811,701.854,179.987,13.652,13.809,125.392,26.937,14.0,<NA>,96.077
1,001004,2000-02-29,54594,23.75,0.342756,0.342756,27181.0,11,1,<NA>,...,713.811,701.854,179.987,13.652,13.809,125.392,26.937,14.0,<NA>,96.077
2,001004,2000-03-31,54594,16.6875,-0.297368,-0.297368,27181.0,11,1,<NA>,...,713.811,701.854,179.987,13.652,13.809,125.392,26.937,14.0,<NA>,96.077
3,001004,2000-04-30,54594,15.0625,-0.092285,-0.097378,26963.0,11,1,<NA>,...,713.811,701.854,179.987,13.652,13.809,125.392,26.937,14.0,<NA>,96.077
4,001004,2000-05-31,54594,13.875,-0.078838,-0.078838,26963.0,11,1,<NA>,...,713.811,701.854,179.987,13.652,13.809,125.392,26.937,14.0,<NA>,96.077


Traitement des NaN:

In [11]:
#Remplacement par 0 
zero_fill = ['xrd', 'capx', 'dp', 'txt', 'xint']
panel[zero_fill] = panel[zero_fill].fillna(0)

In [12]:
#Remplacement par lissage vers l'avant (ou l'arrière si pas dispo)
balance_sheet = ['dltt', 'dlc']
for col in balance_sheet:
    panel[col] = (panel
                  .groupby('gvkey')[col]
                  .ffill()
                  .bfill()
                  .fillna(0))

In [13]:
#Traitement des valeurs manquantes lct (current liabilities) 
print(db_comp['indfmt'].value_counts()) #Contrôle pour vérifier s'il reste des banques

lct_missing = panel.groupby('gvkey')['lct'].apply(lambda x: x.isna().mean())
print("LCT manquants par gvkey :")
print(lct_missing.describe()) #moyenne de manquement correct

bad_lct = lct_missing[lct_missing > 0.8].index.tolist()
print(f"Entreprises LCT >80% manquant : {len(bad_lct)}") #Résultat : 1806, résultat attendu (correspond aux PME sans passif)

panel['lct'] = (panel
                .groupby('gvkey')['lct']
                .ffill()
                .bfill()
                .fillna(0))

indfmt
INDL    178243
Name: count, dtype: Int64
LCT manquants par gvkey :
count    10707.000000
mean         0.170321
std          0.374828
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: lct, dtype: float64
Entreprises LCT >80% manquant : 1806


In [14]:
#Traitement des valeurs manquantes oibdp (Operatinf Income before Depreciation

oibdp_missing = panel.groupby('gvkey')['oibdp'].apply(lambda x: x.isna().mean())
oibdp_missing.describe()

bad_oibdp = oibdp_missing[oibdp_missing == 1].index.tolist() 
print(len(bad_oibdp))

bad_companies = panel[panel['gvkey'].isin(bad_oibdp)][['gvkey', 'conm']].drop_duplicates()
print(bad_companies) 

# 25 entreprises sur des secteurs diversifiés, on fait le choix de les exclures de notre sample

panel = panel[~panel['gvkey'].isin(bad_oibdp)]

panel['oibdp'] = (panel
                .groupby('gvkey')['oibdp']
                .ffill()
                .bfill())

25
          gvkey                          conm
3288     001115             ADAC LABORATORIES
11601    001465         AMERICAN GENERAL CORP
52180    002965           CHEMFAB CORPORATION
60659    003230       COMDISCO HOLDING CO INC
329288   013532           MERCHANTS GROUP INC
394201   017243            TRENWICK GROUP LTD
416259   018792        GAMING & LEISURE PPTYS
426397   019440           LADDER CAPITAL CORP
528048   025573             AXA FINANCIAL INC
540956   026020           ANTEX BIOLOGICS INC
546157   026410  FOUR CORNERS PROPERTY TR INC
548401   026839      PHILLIPS EDISON & CO INC
551608   027394         PARK HOTELS & RESORTS
615451   030165   ASSURED GUARANTY MUNI HLDGS
671110   035305                  VERICITY INC
689844   038753            ENACT HOLDINGS INC
699772   040092           ARCHER AVIATION INC
714138   061302        ARCH CAPITAL GROUP LTD
818852   065175    CORSAIR COMMUNICATIONS INC
899347   117701                  ALLAIRE CORP
925640   122435                

In [15]:
#Traitement des valeurs manquantes xsga (Selling, General and Administrative Expenses)

# xsga = Revenue - COGS - Depreciation - Op. Income

panel['xsga'] = panel['xsga'].fillna(panel['revt'] - panel['cogs'] - panel['oibdp'])


In [16]:
#Traitement des valeurs manquantes oancf (Operating Activity - Net Cash Flow)

oancf_missing = panel.groupby('gvkey')['oancf'].apply(lambda x: x.isna().mean())
oancf_missing.describe()

bad_oancf = oancf_missing[oancf_missing == 1].index.tolist() 
print(len(bad_oancf))

bad_companies1 = panel[panel['gvkey'].isin(bad_oancf)][['gvkey', 'conm']].drop_duplicates() #on s'aperçoit que le filtre INDL au moment de la query n'a pas bien fonctionné car la 
                                                                                            #plupart des entreprises sont ici des banques
print(bad_companies1) 

#Par soucis de simplicité, et comme ce sont majoritairement des banques, on drop 

panel = panel[~panel['gvkey'].isin(bad_oancf)]

#Le peu de valeurs manquantes restant (407) sont lissées

panel['oancf'] = (panel
                .groupby('gvkey')['oancf']
                .ffill()
                .bfill())


233
          gvkey                        conm
27477    001998               BANK ONE CORP
28131    002007        FIRST BANKS AMER INC
101069   004682          U S BANCORP/DE-OLD
101982   004708               BANCWEST CORP
102737   004742    FIRST VIRGINIA BANKS INC
...         ...                         ...
962443   133988         PORT FINANCIAL CORP
971036   137378    FIRST SHARES BANCORP INC
971575   137531    DUTCHFORK BANCSHARES INC
974925   138128          PACIFIC UNION BANK
1039908  157973  FIRST NATL BANKSHRS FL INC

[233 rows x 2 columns]


In [17]:
#Traitement des valeurs manquantes ebit 

# EBIT = OIBDP - DP

panel['ebit'] = panel['ebit'].fillna(panel['oibdp'] - panel['dp'])

In [18]:
#Traitement des valeurs manquantes csho (Common Share Outstanding)

# csho (compustat) peut être rapproché à shrout (CRSP), seulement dans la documentation respective on voit que l'unité est traité différemment

# csho = shrout / 1000 

panel['csho'] = panel['csho'].fillna(panel['shrout'] / 1000) 

#l'utilité de garder les deux peuvent être discuté. Cependant, en réalité elles sont différentes d'un point de vue comptable (ici on a simplifié pour le peu de valeur manquantes (400)

In [49]:
#Traitement des valeurs manquantes des returns et prise en compte du Survivorship Bias

panel[panel['ret'].isna()].sort_values('gvkey')

,gvkey,date,permno,prc,ret,retx,shrout,shrcd,exchcd,dlret,...,cogs,at,dltt,dlc,che,lct,csho,prcc_f,xrd,xsga
734,001045,2013-12-31,21020,25.25,<NA>,<NA>,260000.0,11,3,<NA>,...,29511.0,43771.0,16196.0,1708.0,8077.0,13435.0,697.475,53.63,0.0,6554.0
3339,001117,2005-10-31,10779,5.16,<NA>,<NA>,13071.0,11,2,<NA>,...,13.296,31.116,0.0,0.0,5.283,3.298,13.135,7.31,2.418,9.151
4248,001164,2002-06-30,11042,<NA>,<NA>,<NA>,2962836.0,11,3,<NA>,...,8120.0,91901.0,24533.0,172.0,1409.0,5915.0,2960.671,14.08,0.0,5145.0
4249,001164,2004-07-31,90284,15.28,<NA>,<NA>,314856.0,11,3,<NA>,...,13253.0,17060.0,5909.0,24.0,5504.0,6203.0,319.558,20.16,0.0,5272.0
4431,001177,2000-12-31,88845,41.0625,<NA>,<NA>,141484.0,11,1,<NA>,...,26157.0,47445.7,0.0,1592.2,17516.8,10003.4,142.619,41.0625,0.0,3844.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1189338,333070,2023-09-30,24100,13.12,<NA>,<NA>,5690.0,11,3,<NA>,...,85.582,212.645,1.232,94.439,70.617,187.269,5.698,20.52,0.0,50.719
1189354,339965,2020-09-30,19654,251.0,<NA>,<NA>,36209.0,11,1,<NA>,...,232.762,5921.739,184.887,19.65,3908.064,789.264,287.918,272.45,237.946,893.398
1189442,345764,2022-01-31,22660,6.515,<NA>,<NA>,21561.0,11,3,<NA>,...,1.151,8.665,0.844,0.0,3.476,2.4,20.192,4.0,2.53,10.845
1189478,345920,2020-12-31,20194,52.58,<NA>,<NA>,33500.0,11,3,<NA>,...,271.793,275.795,15.61,4.447,76.955,48.7,33.5,52.58,0.0,58.492


In [19]:
panel.isna().sum()

gvkey               0
date                0
permno              0
prc                38
ret              5138
retx             5138
shrout              0
shrcd               0
exchcd              0
dlret         1181060
dlstcd        1177992
avail_date          0
fyear               0
conm                0
indfmt              0
ni                  0
ceq                 0
oancf               0
sale                0
ebit                0
oibdp               0
dp                  0
txt                 0
xint                0
capx                0
revt                0
cogs                0
at                  0
dltt                0
dlc                 0
che                 0
lct                 0
csho                0
prcc_f            258
xrd                 0
xsga                0
dtype: int64